<a href="https://colab.research.google.com/github/PauloFernandes26/AD2526/blob/main/AD2526_P8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

Python version:  sys.version_info(major=3, minor=12, micro=13, releaselevel='final', serial=0)
TF version:  2.19.0
Keras version:  3.13.2
GPU is available


**1. Obtaining and Preprocessing the Dataset**

In [2]:
# Download IMDB dataset from Keras: https://keras.io/api/datasets/imdb/
# Reviews are already preprocessed and ready to use

# The raw dataset is available here: https://ai.stanford.edu/~amaas/data/sentiment/

tf.random.set_seed(42)

# Check the documentation of the load_data function

# These two features are important
max_features = 10000
common_words = 10

# Load train and test datasets
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features, skip_top=common_words)

# Load a dictionary that will be used to decode reviews
word_index = keras.datasets.imdb.get_word_index()

print('Train dataset size: ', len(x_train))
print('Test dataset size: ', len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train dataset size:  25000
Test dataset size:  25000


**Quiz 1**

Consult the documentation to answer the following questions:

What preprocessing operations have been done to the original reviews?

The load_data() method has 2 parameters. Why are they important to prepare the dataset?

What is the size of the total vocabulary of the IMDB dataset?

1.   What preprocessing operations have been done to the original reviews?

2.   The load_data() method has 2 parameters. Why are they important to prepare the dataset?

3.   What is the size of the total vocabulary of the IMDB dataset?



**1.1. Consult Some Reviews**

In [3]:
# Visualizing some reviews

# By relying on the retrieved dictionary, it is also possible to visualize the decoded review

# Labels: 0(Bad), 1(Good)

# Choose review
review = 0

print("Word count: " ,len(x_train[review]))
print(x_train[review])

tam = len(x_train[review])
print('Label ', y_train[review])


id_to_word = {id_ + 3: word for word, id_ in word_index.items()}
for id_, token in enumerate(("<pad>", "<sos>", "<unk>")):
    id_to_word[id_] = token
" ".join([id_to_word[id_] for id_ in x_train[review][:tam]])

Word count:  218
[2, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 2, 173, 36, 256, 2, 25, 100, 43, 838, 112, 50, 670, 2, 2, 35, 480, 284, 2, 150, 2, 172, 112, 167, 2, 336, 385, 39, 2, 172, 4536, 1111, 17, 546, 38, 13, 447, 2, 192, 50, 16, 2, 147, 2025, 19, 14, 22, 2, 1920, 4613, 469, 2, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 2, 22, 17, 515, 17, 12, 16, 626, 18, 2, 2, 62, 386, 12, 2, 316, 2, 106, 2, 2, 2223, 5244, 16, 480, 66, 3785, 33, 2, 130, 12, 16, 38, 619, 2, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 2, 22, 12, 215, 28, 77, 52, 2, 14, 407, 16, 82, 2, 2, 2, 107, 117, 5952, 15, 256, 2, 2, 2, 3766, 2, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 2, 2, 2, 1029, 13, 104, 88, 2, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 2, 194, 7486, 18, 2, 226, 22, 21, 134, 476, 26, 480, 2, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 2, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 2, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Label  1


"<unk> this film was just brilliant casting location scenery story direction everyone's really suited <unk> part they played <unk> you could just imagine being there robert <unk> <unk> an amazing actor <unk> now <unk> same being director <unk> father came from <unk> same scottish island as myself so i loved <unk> fact there was <unk> real connection with this film <unk> witty remarks throughout <unk> film were great it was just brilliant so much that i bought <unk> film as soon as it was released for <unk> <unk> would recommend it <unk> everyone <unk> watch <unk> <unk> fly fishing was amazing really cried at <unk> end it was so sad <unk> you know what they say if you cry at <unk> film it must have been good <unk> this definitely was also <unk> <unk> <unk> two little boy's that played <unk> <unk> <unk> norman <unk> paul they were just brilliant children are often left out <unk> <unk> <unk> list i think because <unk> stars that play them all grown up are such <unk> big profile for <unk> 

**1.2. Trim Reviews**

In [4]:
# Trim reviews and keep just the last maxlen words

# The classifier will perform sentiment anaslysis just considering these last words of the reviews
# https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/sequence/pad_sequences

maxlen = 20

x_trainP = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_testP = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

In [5]:
# Visualizing reviews after trimming

# Choose a review
review = 10

print("Word count: " ,len(x_trainP[review]))

print('Label ', y_train[review])
id_to_word = {id_ + 3: word for word, id_ in word_index.items()}
for id_, token in enumerate(("<pad>", "<sos>", "<unk>")):
    id_to_word[id_] = token
" ".join([id_to_word[id_] for id_ in x_trainP[review][:tam]])

Word count:  20
Label  1


'horrifying overall <unk> <unk> <unk> truly great horror film <unk> one <unk> <unk> best <unk> <unk> decade highly recommended viewing'

**2. Classifiers for Sentiment Analysis**

**2.1 Model A - Multilayer Perceptron (MLP)**

In [6]:
# Baseline Feed-Forward Neural Network

# Complete the model with two hidden layers, each with 20 nodes (default parameters)
# The last layer should have 1 node with sigmoid activation

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

inputs = keras.Input(shape=[maxlen, 1])
x = keras.layers.Flatten()(inputs)

### Complete the Missing layers ###

x = keras.layers.Dense(20, activation='relu')(x)
x = keras.layers.Dense(20, activation='relu')(x)


output = keras.layers.Dense(1, activation='sigmoid')(x)

modelA = keras.Model(inputs, output)

modelA.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 861 (3.36 KB)

 Trainable params: 861 (3.36 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
modelA.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

history = modelA.fit(x_trainP, y_train, epochs=10, validation_split=0.2)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5011 - loss: 57.4170 - val_accuracy: 0.4998 - val_loss: 20.8879
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5000 - loss: 11.0967 - val_accuracy: 0.5016 - val_loss: 3.9199
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4963 - loss: 3.4647 - val_accuracy: 0.5032 - val_loss: 1.7907
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4981 - loss: 2.0995 - val_accuracy: 0.4954 - val_loss: 2.2777
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5037 - loss: 1.7363 - val_accuracy: 0.5080 - val_loss: 2.6159
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5041 - loss: 1.5722 - val_accuracy: 0.4964 - val_loss: 1.3942
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5048 - loss: 1.4680 - val_accuracy: 0.5002 - val_loss: 1.1245
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5070 - loss: 1.3408 - val_accuracy:

In [8]:
# Evaluate ModelA on the test set

modelA.evaluate(x_testP, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4959 - loss: 0.7676


[0.7676368951797485, 0.49592000246047974]

**2.2. Model B: MultiLayer Perceptron with Word Embedding**

In [9]:
# Add a trainable embedding layer. The embedding should habe 8 dimensions
# The remaining layers should be identical to Model A

# https://www.tensorflow.org/tutorials/text/word_embeddings
# https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

output_emb = 8

inputs = keras.Input(shape=[maxlen])
emb = keras.layers.Embedding(max_features,  output_emb)(inputs)
x = keras.layers.Flatten()(emb)

### Complete the Missing layers ###

x = keras.layers.Dense(20, activation='relu')(x)
x = keras.layers.Dense(20, activation='relu')(x)


output = keras.layers.Dense(1, activation='sigmoid')(x)

modelB = keras.Model(inputs, output)

modelB.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 20, 8)          │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20)             │         3,220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,661 (326.80 KB)

 Trainable params: 83,661 (326.80 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
modelB.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

history = modelB.fit(x_trainP, y_train,
                      epochs=10,
                      validation_split=0.2)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6223 - loss: 0.6240 - val_accuracy: 0.7384 - val_loss: 0.5169
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8191 - loss: 0.3944 - val_accuracy: 0.7400 - val_loss: 0.5663
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9058 - loss: 0.2393 - val_accuracy: 0.7186 - val_loss: 0.7420
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9592 - loss: 0.1207 - val_accuracy: 0.6950 - val_loss: 1.0608
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9860 - loss: 0.0504 - val_accuracy: 0.6856 - val_loss: 1.3765
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9951 - loss: 0.0217 - val_accuracy: 0.6930 - val_loss: 1.5381
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9980 - loss: 0.0097 - val_accuracy: 0.7002 - val_loss: 1.7657
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9992 - loss: 0.0041 - val_accuracy: 0.

In [11]:
# Evaluate Model B on the test set

modelB.evaluate(x_testP, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7002 - loss: 2.1466


[2.1466457843780518, 0.700160026550293]

**2.3. Model C: Recurrent Neural Network (RNN) with Word Embedding**

In [12]:
# The feed-forward cells from hidden layers are replaced by recurrent GRU cells
# Everything else is identical to ModelB

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

output_emb = 8

inputs = keras.Input(shape=[maxlen])
emb = keras.layers.Embedding(max_features,  output_emb)(inputs)

### Complete the Missing layers ###

x= keras.layers.GRU(20, return_sequences=True)(emb)
x= keras.layers.GRU(20)(x)



output = keras.layers.Dense(1, activation='sigmoid')(x)

modelC = keras.Model(inputs, output)

modelC.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 20, 8)          │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 20)         │         1,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 20)             │         2,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,341 (329.46 KB)

 Trainable params: 84,341 (329.46 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
modelC.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

history = modelC.fit(x_trainP, y_train,
                      epochs=10,
                      validation_split=0.2)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7047 - loss: 0.5514 - val_accuracy: 0.7480 - val_loss: 0.4928
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8065 - loss: 0.4163 - val_accuracy: 0.7488 - val_loss: 0.5081
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8400 - loss: 0.3637 - val_accuracy: 0.7456 - val_loss: 0.5504
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8615 - loss: 0.3286 - val_accuracy: 0.7368 - val_loss: 0.6006
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8782 - loss: 0.2983 - val_accuracy: 0.7322 - val_loss: 0.6647
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8974 - loss: 0.2642 - val_accuracy: 0.7248 - val_loss: 0.7510
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9144 - loss: 0.2284 - val_accuracy: 0.7260 - val_loss: 0.8543
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9307 - loss: 0.1980 - val_accuracy: 0.

In [14]:
# Evaluate Model C on the test set

modelC.evaluate(x_testP, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7060 - loss: 0.9243


[0.924285888671875, 0.7059599757194519]

**2.4. Model D: Recurrent Neural Network with Pretrained Embedding**

In [15]:
# Using a pretrained embedding

#Two options to obtain the embedding

# Option 1: Direct download from Stanford

# In this example we will adopt the GloVe with 50 dimensions: https://nlp.stanford.edu/projects/glove/

!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

!rm glove.6B.100d.txt
!rm glove.6B.200d.txt
!rm glove.6B.300d.txt
!rm glove.6B.zip


# Option 2: Upload the file emb.zip to the working directory

#!unzip -q emb.zip

#!rm emb.zip


embeddings_index = {}
f = open(os.path.join('glove.6B.50d.txt'))
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

# Create the embedding matrix with max_words lines and 50 columns (the embedding dimension)

embedding_dim = 50
max_words = max_features
embedding_matrix = np.zeros((max_words, embedding_dim)) # matriz com zeros

# Fill the matrix with the values of the pretrained embedding

for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if i < max_words:
        if embedding_vector is not None:
            # Words not found in embedding index will be all-zeros.
            embedding_matrix[i] = embedding_vector


--2026-04-14 20:12:20--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-14 20:12:20--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.00MB/s    in 2m 39s  

2026-04-14 20:14:59 (5.17 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Found 400000 word vectors.


In [16]:
# Create model D, which is identical to model C with the exception of the embedding dimension

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

inputs = keras.Input(shape=[maxlen])
emb = keras.layers.Embedding(max_words, embedding_dim)(inputs)


### Complete the Missing layers ###
x= keras.layers.GRU(20, return_sequences=True)(emb)
x= keras.layers.GRU(20)(x)

output = keras.layers.Dense(1, activation='sigmoid')(x)

modelD = keras.Model(inputs, output)

# Store the pretrained embedding values in the embedding layer
# Freeze the weights, so that they do not change during training

modelD.layers[1].set_weights([embedding_matrix])
modelD.layers[1].trainable = False

modelD.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 20, 50)         │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 20)         │         4,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 20)             │         2,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 506,861 (1.93 MB)

 Trainable params: 6,861 (26.80 KB)

 Non-trainable params: 500,000 (1.91 MB)

In [17]:
modelD.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])


history = modelD.fit(x_trainP, y_train,
                      epochs=10,
                      validation_split=0.2)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.5419 - loss: 0.6870 - val_accuracy: 0.5732 - val_loss: 0.6736
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5856 - loss: 0.6664 - val_accuracy: 0.6056 - val_loss: 0.6515
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6259 - loss: 0.6391 - val_accuracy: 0.6316 - val_loss: 0.6317
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6536 - loss: 0.6137 - val_accuracy: 0.6406 - val_loss: 0.6190
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6689 - loss: 0.5948 - val_accuracy: 0.6520 - val_loss: 0.6127
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6811 - loss: 0.5804 - val_accuracy: 0.6546 - val_loss: 0.6094
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6896 - loss: 0.5686 - val_accuracy: 0.6580 - val_loss: 0.6075
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6985 - loss: 0.5584 - val_accuracy: 0.

In [18]:
# Evaluate modelD in the test set

modelD.evaluate(x_testP, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6736 - loss: 0.5994


[0.5993970036506653, 0.6736400127410889]

**Quiz**

1. Analyse the performance of the different models
2. Present justifications for the comparative accuracy of the four models.

**3.	Text Preprocessing / Hyperparameter Analysis / Overfitting**

Several preprocessing operations and hyperparameters are used in this example and they can have a relevant impact in the performance of the models. Create a new experiment by changing one of the following options and analyze the impact on performance.

3.1.	Reviews Preprocessing

a)	Removal of frequent words from the dataset.

b)	Vocabulary size

c)	Is the relative order of the words relevant for classifying a review?

d)	Review trimming: length and/or beginning vs. end.


3.2.	Hyperparameters

a)	Embedding dimensions

b)	Embedding type: pretrained vs. trained from scratch

c)	Neural network architecture: layers, nodes, cell type, …

d)	Other relevant hyperparameters



3.3.	Overfitting / Regularization

Add techniques to fight overfitting / to promote regularization.



In [23]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================================================
# 1) Reprodutibilidade
# =========================================================
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

# =========================================================
# 2) Hiperparâmetros
# =========================================================
max_features = 20000      # vocabulário maior
common_words = 10         # ignora as palavras mais frequentes
maxlen = 200              # 200 costuma ser um bom equilíbrio
embedding_dim = 128       # embedding mais rico
batch_size = 64
epochs = 10               # como pediste

# =========================================================
# 3) Carregar o dataset IMDb
# =========================================================
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_features,
    skip_top=common_words
)

# =========================================================
# 4) Padding das sequências
# =========================================================
# padding='post' -> adiciona zeros no fim
# truncating='post' -> corta no fim, mantendo o início da review
x_trainP = pad_sequences(x_train, maxlen=maxlen, padding='post', truncating='post')
x_testP = pad_sequences(x_test, maxlen=maxlen, padding='post', truncating='post')

print("x_trainP shape:", x_trainP.shape)
print("x_testP shape:", x_testP.shape)

# =========================================================
# 5) Modelo C melhorado
# Continua a ser um modelo GRU, mas mais forte
# =========================================================
inputs = keras.Input(shape=(maxlen,))

# Embedding treinável
emb = keras.layers.Embedding(
    input_dim=max_features,
    output_dim=embedding_dim
)(inputs)

# Dropout nos embeddings para ajudar na generalização
x = keras.layers.SpatialDropout1D(0.2)(emb)

# 1ª camada GRU bidirecional
x = keras.layers.Bidirectional(
    keras.layers.GRU(
        64,
        return_sequences=True,
        dropout=0.2
    )
)(x)

# 2ª camada GRU bidirecional
x = keras.layers.Bidirectional(
    keras.layers.GRU(
        32,
        dropout=0.2
    )
)(x)

# Camada densa intermédia
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)

# Saída binária
output = keras.layers.Dense(1, activation='sigmoid')(x)

modelC = keras.Model(inputs, output)

modelC.summary()

# =========================================================
# 6) Compilar o modelo
# =========================================================
modelC.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

# =========================================================
# 7) Callbacks
# =========================================================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5
    )
]

# =========================================================
# 8) Treino
# =========================================================
history = modelC.fit(
    x_trainP,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

# =========================================================
# 9) Avaliação final
# =========================================================
test_loss, test_acc = modelC.evaluate(x_testP, y_test, verbose=1)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

x_trainP shape: (25000, 200)
x_testP shape: (25000, 200)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 200, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 128)       │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,669,825 (10.18 MB)

 Trainable params: 2,669,825 (10.18 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.6927 - loss: 0.5574 - val_accuracy: 0.8380 - val_loss: 0.3885 - learning_rate: 0.0010
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.8729 - loss: 0.3196 - val_accuracy: 0.8482 - val_loss: 0.3706 - learning_rate: 0.0010
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.9198 - loss: 0.2197 - val_accuracy: 0.8518 - val_loss: 0.3824 - learning_rate: 0.0010
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.9557 - loss: 0.1279 - val_accuracy: 0.8596 - val_loss: 0.4466 - learning_rate: 5.0000e-04
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.8217 - loss: 0.4289

Test Loss: 0.4289
Test Accuracy: 0.8217
